# 1. Setup

**Zweck dieses Notebooks:** die (teure) Clique-Discovery -- für jeden Kadenz-Schritt und Basket die aktuell persistenteste Clique per `HelperFunctions.get_persistent_clusters` finden -- EINMAL für den gesamten verfügbaren Zeitraum ausführen und als Parquet-Dateien ablegen (`data/processed/cliques/`). Nachgelagerte Notebooks (z.B. `03_signal_generation/01_forward_return_screening.ipynb`, die Backtest-Engine) lesen diese Dateien ein statt die Discovery selbst zu wiederholen.

Erzeugt **sechs gleichwertige Datensätze** -- {roh, residualisiert} x {6h, 12h, 24h} Discovery-Kadenz (`cliques_raw_6h.parquet` ... `cliques_residual_24h.parquet`). Es gibt bewusst keine einzelne "Produktions"-Datei mehr mit einer fest gewählten Kadenz pro Basket -- die Kadenz-Wahl war früher eine Churn-/Lebensdauer-Heuristik, die sich als nicht überzeugend erwies (siehe `02_pointclouds.ipynb`, Abschnitt 4). Der eigentliche Kadenz-Vergleich passiert stattdessen später per echtem Backtest (`04_backtest/`).

Nutzt dieselben kalibrierten Parameter wie `02_pointclouds.ipynb` (roh) und `03_residual_pointclouds.ipynb` (residualisiert) -- Abschnitt 4 dort: Lookback fix pro Basket, Threshold pro (Basket, Kadenz). **Wichtig:** wird dort neu kalibriert, muss dieses Notebook erneut ausgeführt werden, sonst sind die Caches veraltet (die kalibrierten Werte werden mit in die Cache-Metadaten geschrieben, damit lesende Notebooks das prüfen können, siehe Abschnitt 6).


In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy as np
import pandas as pd

from src.config import (
    ASSET_DAILY_PATH,
    ASSET_ID_COL,
    ASSETS_PATH,
    CLOSE_COL,
    DATE_COL,
    MARKET_SYMBOL_COL,
    MIN_VOLUME_24H_USD,
    OHLCV_DIR,
    PROCESSED_DIR,
    PROJECT_ROOT,
    TIMESTAMP_COL,
    VOLUME_COL,
)
from src.clustering.helpers import HelperFunctions, STABLECOIN_SYMBOLS

helper = HelperFunctions()

print("Imports OK")

Imports OK


# 2. Daten laden & Basket-Konfiguration

In [2]:
# --- Daten laden & vorverarbeiten (identisch zu 02_pointclouds.ipynb) ---
assets_df = pd.read_parquet(ASSETS_PATH)[["asset_id", "symbol"]].drop_duplicates()
id_to_symbol = dict(zip(assets_df["asset_id"], assets_df["symbol"]))
sym_to_id = dict(zip(assets_df["symbol"], assets_df["asset_id"]))

excluded_ids_stablecoins: set[int] = set(
    assets_df.loc[assets_df["symbol"].isin(STABLECOIN_SYMBOLS), "asset_id"].tolist()
)

ohlcv_files = sorted(OHLCV_DIR.glob("*.parquet"))
raw = pd.concat([pd.read_parquet(f) for f in ohlcv_files], ignore_index=True)
raw["base_symbol"] = raw[MARKET_SYMBOL_COL].str.replace("USDT$", "", regex=True)
raw = raw.merge(assets_df.rename(columns={"symbol": "base_symbol"}), on="base_symbol", how="left")
raw = raw.dropna(subset=["asset_id"])
raw["asset_id"] = raw["asset_id"].astype(int)
raw[TIMESTAMP_COL] = pd.to_datetime(raw[TIMESTAMP_COL], utc=True)
raw[DATE_COL] = raw[TIMESTAMP_COL].dt.normalize()
ohlcv_df = raw[[TIMESTAMP_COL, DATE_COL, ASSET_ID_COL, CLOSE_COL]].copy()

data_min = ohlcv_df[DATE_COL].min().date()
data_max = ohlcv_df[DATE_COL].max().date()

universe_df = pd.read_parquet(ASSET_DAILY_PATH)
universe_df[DATE_COL] = pd.to_datetime(universe_df[DATE_COL], utc=True)
all_asset_ids_local = set(universe_df[ASSET_ID_COL].unique())

print(f"Zeitraum: {data_min} → {data_max}")

Zeitraum: 2024-07-23 → 2026-07-22


In [3]:
# --- Basket-Konfiguration (identisch zu 02_pointclouds.ipynb) ---
POINTCLOUD_LOOKBACK_DAYS = 7
POINTCLOUD_BASKET_UPPER_BOUNDS = [100_000_000, 250_000_000, 1_000_000_000]

rolling_volume_df = helper.compute_rolling_volume_table(
    universe_df, id_column=ASSET_ID_COL, date_column=DATE_COL,
    volume_column=VOLUME_COL, lookback_days=POINTCLOUD_LOOKBACK_DAYS,
)
basket_labels = [f"Basket {n}" for n in range(1, len(POINTCLOUD_BASKET_UPPER_BOUNDS) + 2)]

# 3. Kalibrierte Parameter (aus 02_tda_clustering übernommen)

Direkt aus `02_pointclouds.ipynb` (roh) und `03_residual_pointclouds.ipynb` (residualisiert) Abschnitt 4 kopiert -- gleiches Muster wie überall sonst, wo diese Werte gebraucht werden.

In [4]:
# --- Roh (02_pointclouds.ipynb, Abschnitt 4) ---
# Lookback ist FIX pro Basket (per Jaccard-Stabilität gewählt, unabhängig
# von der Kadenz). Threshold ist PRO (Basket, Kadenz) kalibriert (per
# Persistenz-Histogramm bei der jeweiligen Kadenz -- die Stichprobe der
# Persistenzwerte hängt von der Abtastdichte ab). Ersetzt die frühere
# BEST_CADENCE_HOURS_RAW-Wahl (Churn-/Lebensdauer-Heuristik, 24h/12h/12h/12h) --
# die driftete bei allen Baskets ähnlich stark mit der Kadenz und trug die
# Basket-Differenzierung nicht überzeugend. Stattdessen werden unten für
# ALLE drei Kadenzen (6h/12h/24h) gleichwertige Cache-Dateien erzeugt, der
# eigentliche Vergleich/Entscheid kommt per echtem Backtest (04_backtest/).
#
BEST_LOOKBACK_RAW: dict[str, int] = {
    "Basket 1": 7,
    "Basket 2": 10,
    "Basket 3": 15,
    "Basket 4": 5,
}
BEST_THRESHOLD_RAW: dict[str, dict[int, float]] = {
    "Basket 1": {6: 0.1999, 12: 0.1982, 24: 0.1916},
    "Basket 2": {6: 0.1093, 12: 0.1091, 24: 0.1085},
    "Basket 3": {6: 0.1266, 12: 0.1273, 24: 0.1272},
    "Basket 4": {6: 0.2545, 12: 0.2545, 24: 0.2570},
}
CADENCE_VARIANTS_HOURS = [6, 12, 24]
EPSILON_CAP_RAW = 1.0

# --- Residualisiert (03_residual_pointclouds.ipynb, Abschnitt 4) ---
FACTOR_COLUMNS_RESIDUAL = ["BTC", "ETH"]
REGRESSION_LOOKBACK_DAYS = 60
BEST_LOOKBACK_RESIDUAL: dict[str, int] = {
    "Basket 1": 7,
    "Basket 2": 10,
    "Basket 3": 15,
    "Basket 4": 5,
}
BEST_THRESHOLD_RESIDUAL: dict[str, dict[int, float]] = {
    "Basket 1": {6: 0.1618, 12: 0.1604, 24: 0.1605},
    "Basket 2": {6: 0.0918, 12: 0.0918, 24: 0.0916},
    "Basket 3": {6: 0.1127, 12: 0.1139, 24: 0.1150},
    "Basket 4": {6: 0.2053, 12: 0.2082, 24: 0.2023},
}
EPSILON_CAP_RESIDUAL = 1.3

factor_asset_ids: set[int] = {sym_to_id[s] for s in FACTOR_COLUMNS_RESIDUAL if s in sym_to_id}

print("Roh:            Lookback", BEST_LOOKBACK_RAW, "| Kadenzen(h)", CADENCE_VARIANTS_HOURS, "| epsilon_cap", EPSILON_CAP_RAW)
print("Residualisiert: Lookback", BEST_LOOKBACK_RESIDUAL, "| Kadenzen(h)", CADENCE_VARIANTS_HOURS, "| epsilon_cap", EPSILON_CAP_RESIDUAL)


Roh:            Lookback {'Basket 1': 7, 'Basket 2': 10, 'Basket 3': 15, 'Basket 4': 5} | Kadenzen(h) [6, 12, 24] | epsilon_cap 1.0
Residualisiert: Lookback {'Basket 1': 7, 'Basket 2': 10, 'Basket 3': 15, 'Basket 4': 5} | Kadenzen(h) [6, 12, 24] | epsilon_cap 1.3


# 4. Cliquen berechnen & cachen: roh

**Achtung, das ist die teure Zelle:** volle Persistenzberechnung für JEDEN Kadenz-Schritt im gesamten Datenbereich (nicht nur den Kalibrierungszeitraum -- dieser Cache soll auch später für den Backtest wiederverwendbar sein) x 4 Baskets x 3 Kadenzen (6h/12h/24h). Läuft nur bei `RUN_DISCOVERY=True`; danach stehen die Ergebnisse in den drei Parquet-Dateien und müssen nicht mehr neu gerechnet werden, bis sich die kalibrierten Parameter oben ändern.

`DATE_STRIDE=1` (jeder einzelne Kadenz-Schritt) ist Standard, weil der Cache möglichst vollständig/allgemein wiederverwendbar sein soll -- nachgelagerte Notebooks können beim Einlesen selbst weiter ausdünnen, wenn sie es eiliger haben.

Erzeugt `cliques_raw_6h.parquet`, `cliques_raw_12h.parquet`, `cliques_raw_24h.parquet` -- alle 4 Baskets nutzen INNERHALB einer Datei dieselbe Kadenz (im Unterschied zur früheren, entfernten Produktions-Mischkadenz), nur der Threshold unterscheidet sich zwischen den Baskets (siehe Abschnitt 3).


In [5]:
RUN_DISCOVERY = True
DATE_STRIDE = 1  # 1 = jeder Kadenz-Schritt; nur zum schnellen Testen hochsetzen
CLIQUES_DIR = PROCESSED_DIR / "cliques"
CLIQUES_DIR.mkdir(parents=True, exist_ok=True)

cliques_by_variant: dict[str, pd.DataFrame] = {}

if RUN_DISCOVERY:
    for h in CADENCE_VARIANTS_HOURS:
        variant_dates = pd.date_range(data_min, data_max, freq=f"{h}h", tz="UTC")[::DATE_STRIDE]
        best_threshold_h = {label: BEST_THRESHOLD_RAW[label][h] for label in basket_labels}

        cliques_variant = HelperFunctions.discover_cliques_over_dates(
            ohlcv_df=ohlcv_df, universe_df=universe_df, rolling_volume_df=rolling_volume_df,
            id_column=ASSET_ID_COL, date_column=DATE_COL,
            basket_labels=basket_labels, basket_upper_bounds=POINTCLOUD_BASKET_UPPER_BOUNDS,
            exclude_ids_for_search=excluded_ids_stablecoins, all_asset_ids=all_asset_ids_local,
            best_lookback=BEST_LOOKBACK_RAW, best_threshold=best_threshold_h, epsilon_cap=EPSILON_CAP_RAW,
            point_cloud_fn=HelperFunctions.build_point_cloud,
            dates=variant_dates,
        )
        cliques_by_variant[f"raw_{h}h"] = cliques_variant

        variant_path = CLIQUES_DIR / f"cliques_raw_{h}h.parquet"
        cliques_variant.to_parquet(variant_path, index=False)
        json.dump(
            {
                "variant": f"raw_{h}h",
                "best_lookback": BEST_LOOKBACK_RAW,
                "best_threshold": best_threshold_h,
                "cadence_hours": h,
                "epsilon_cap": EPSILON_CAP_RAW,
                "basket_upper_bounds": POINTCLOUD_BASKET_UPPER_BOUNDS,
                "date_range": [str(data_min), str(data_max)],
                "date_stride": DATE_STRIDE,
            },
            open(CLIQUES_DIR / f"cliques_raw_{h}h.meta.json", "w"),
            indent=2,
        )
        print(f"{h:>2}h: {len(cliques_variant)} Cliquen über {variant_dates.size} Zeitpunkte x {len(basket_labels)} Baskets "
              f"-> {variant_path.relative_to(PROJECT_ROOT)}")
else:
    print("RUN_DISCOVERY=False -- roh Kadenz-Caches nicht neu berechnet.")


 6h: 7152 Cliquen über 2917 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_raw_6h.parquet


12h: 3580 Cliquen über 1459 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_raw_12h.parquet


24h: 1799 Cliquen über 730 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_raw_24h.parquet


# 5. Cliquen berechnen & cachen: residualisiert

`_residual_point_cloud_fn` übersetzt zwischen der `point_cloud_fn`-Signatur und `HelperFunctions.build_residual_point_cloud` -- identisches Wrapper-Muster wie in `03_residual_pointclouds.ipynb` Abschnitt 3.

Gleiches Muster wie Abschnitt 4 (roh): erzeugt `cliques_residual_6h.parquet`, `cliques_residual_12h.parquet`, `cliques_residual_24h.parquet`, mit dem je (Basket, Kadenz) kalibrierten Threshold aus `03_residual_pointclouds.ipynb`.


In [6]:
def _residual_point_cloud_fn(*, ohlcv_df, universe_df, end_date, window_size_days, exclude_asset_ids):
    return HelperFunctions.build_residual_point_cloud(
        ohlcv_df=ohlcv_df, universe_df=universe_df, end_date=end_date,
        id_to_symbol=id_to_symbol, factor_columns=FACTOR_COLUMNS_RESIDUAL,
        regression_lookback_days=REGRESSION_LOOKBACK_DAYS,
        topological_lookback_days=window_size_days,
        exclude_asset_ids=exclude_asset_ids,
    )


if RUN_DISCOVERY:
    for h in CADENCE_VARIANTS_HOURS:
        variant_dates = pd.date_range(data_min, data_max, freq=f"{h}h", tz="UTC")[::DATE_STRIDE]
        best_threshold_h = {label: BEST_THRESHOLD_RESIDUAL[label][h] for label in basket_labels}

        cliques_variant = HelperFunctions.discover_cliques_over_dates(
            ohlcv_df=ohlcv_df, universe_df=universe_df, rolling_volume_df=rolling_volume_df,
            id_column=ASSET_ID_COL, date_column=DATE_COL,
            basket_labels=basket_labels, basket_upper_bounds=POINTCLOUD_BASKET_UPPER_BOUNDS,
            exclude_ids_for_search=excluded_ids_stablecoins | factor_asset_ids, all_asset_ids=all_asset_ids_local,
            best_lookback=BEST_LOOKBACK_RESIDUAL, best_threshold=best_threshold_h, epsilon_cap=EPSILON_CAP_RESIDUAL,
            point_cloud_fn=_residual_point_cloud_fn,
            dates=variant_dates,
        )
        cliques_by_variant[f"residual_{h}h"] = cliques_variant

        variant_path = CLIQUES_DIR / f"cliques_residual_{h}h.parquet"
        cliques_variant.to_parquet(variant_path, index=False)
        json.dump(
            {
                "variant": f"residual_{h}h",
                "factor_columns": FACTOR_COLUMNS_RESIDUAL,
                "regression_lookback_days": REGRESSION_LOOKBACK_DAYS,
                "best_lookback": BEST_LOOKBACK_RESIDUAL,
                "best_threshold": best_threshold_h,
                "cadence_hours": h,
                "epsilon_cap": EPSILON_CAP_RESIDUAL,
                "basket_upper_bounds": POINTCLOUD_BASKET_UPPER_BOUNDS,
                "date_range": [str(data_min), str(data_max)],
                "date_stride": DATE_STRIDE,
            },
            open(CLIQUES_DIR / f"cliques_residual_{h}h.meta.json", "w"),
            indent=2,
        )
        print(f"{h:>2}h: {len(cliques_variant)} Cliquen über {variant_dates.size} Zeitpunkte x {len(basket_labels)} Baskets "
              f"-> {variant_path.relative_to(PROJECT_ROOT)}")
else:
    print("RUN_DISCOVERY=False -- residual Kadenz-Caches nicht neu berechnet.")


 6h: 4740 Cliquen über 2917 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_residual_6h.parquet


12h: 2364 Cliquen über 1459 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_residual_12h.parquet


24h: 1182 Cliquen über 730 Zeitpunkte x 4 Baskets -> data/processed/cliques/cliques_residual_24h.parquet


# 6. Kurzer Überblick

In [7]:
for name, df in cliques_by_variant.items():
    print(f"--- {name} ---")
    print(f"  {len(df)} Zeilen, {df['discovery_date'].nunique()} Zeitpunkte mit mind. einer Clique")
    print(f"  Cliquen pro Basket:\n{df['basket'].value_counts().sort_index().to_string()}")
    print(f"  Mittlere Cliquen-Grösse: {df['size'].mean():.2f}, mittlere Persistenz: {df['persistence'].mean():.4f}")
    print()


--- raw_6h ---
  7152 Zeilen, 2762 Zeitpunkte mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1    2070
Basket 2    2408
Basket 3    1844
Basket 4     830
  Mittlere Cliquen-Grösse: 7.75, mittlere Persistenz: 0.2638

--- raw_12h ---
  3580 Zeilen, 1382 Zeitpunkte mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1    1040
Basket 2    1209
Basket 3     917
Basket 4     414
  Mittlere Cliquen-Grösse: 7.75, mittlere Persistenz: 0.2637

--- raw_24h ---
  1799 Zeilen, 694 Zeitpunkte mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1    531
Basket 2    605
Basket 3    459
Basket 4    204
  Mittlere Cliquen-Grösse: 7.74, mittlere Persistenz: 0.2632

--- residual_6h ---
  4740 Zeilen, 2311 Zeitpunkte mit mind. einer Clique
  Cliquen pro Basket:
basket
Basket 1     945
Basket 2    1745
Basket 3    1394
Basket 4     656
  Mittlere Cliquen-Grösse: 7.32, mittlere Persistenz: 0.2109

--- residual_12h ---
  2364 Zeilen, 1153 Zeitpunkte mit mind. einer Clique
  Cliquen p